In [ ]:
import pandas as pd
import awswrangler as wr
import time
from datetime import datetime
import gc
import os
import psutil
import boto3

def get_memory_usage():
    """Return the current memory usage of the process in GB"""
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024 / 1024 / 1024
    return memory_gb

print(f"Initial memory usage: {get_memory_usage():.2f} GB")

# List of band columns we want to melt
band_cols = [f"BANDAVAIL{i}" for i in range(1, 11)]

def melt_and_write_chunks(df, chunk_size=50000, timestamp=None):
    """
    Melts the BANDAVAIL columns in chunks and writes each chunk directly to S3
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        
    print(f"Starting streaming melt of dataframe with shape: {df.shape}")
    print(f"Current memory usage: {get_memory_usage():.2f} GB")
    start_time = time.time()
    
    # Calculate number of chunks
    num_chunks = (len(df) + chunk_size - 1) // chunk_size
    print(f"Processing in {num_chunks} chunks of size {chunk_size}")
    
    # Create output path
    output_base = f"s3://thesis--ec331-s3/melted-volume-bids/enriched_volume_bids_melted_{timestamp}"
    
    # Track total rows processed
    total_rows_processed = 0
    all_chunk_files = []
    
    # Process dataframe in chunks
    for i in range(num_chunks):
        chunk_start = i * chunk_size
        chunk_end = min((i + 1) * chunk_size, len(df))
        
        print(f"Processing chunk {i+1}/{num_chunks} (rows {chunk_start} to {chunk_end-1})")
        print(f"Memory before chunk processing: {get_memory_usage():.2f} GB")
        
        # Extract chunk and immediately release the original slice reference
        chunk = df.iloc[chunk_start:chunk_end].copy()
        
        # Identify columns not being melted
        id_vars_cols = [col for col in chunk.columns if col not in band_cols]
        
        # Keep only necessary columns
        chunk = chunk[id_vars_cols + [col for col in band_cols if col in chunk.columns]]
        
        # Melt this chunk
        chunk_start_time = time.time()
        chunk_melted = pd.melt(
            chunk,
            id_vars=id_vars_cols,
            value_vars=[col for col in band_cols if col in chunk.columns],
            var_name="BIDBAND",
            value_name="BIDVOLUME"
        )
        
        # Clean up original chunk to free memory
        del chunk
        gc.collect()
        
        # Extract the band number
        chunk_melted["BIDBAND"] = chunk_melted["BIDBAND"].str.extract(r"BANDAVAIL(\d+)").astype(int)
        
        # Filter out null values to reduce size
        chunk_melted = chunk_melted.dropna(subset=["BIDVOLUME"])
        
        # Count rows in this chunk
        chunk_rows = len(chunk_melted)
        total_rows_processed += chunk_rows
        
        # Write this chunk directly to S3
        chunk_output = f"{output_base}_part{i+1:04d}.parquet"
        write_start = time.time()
        
        try:
            wr.s3.to_parquet(
                df=chunk_melted,
                path=chunk_output,
                index=False,
                compression="snappy"
            )
            all_chunk_files.append(chunk_output)
            write_time = time.time() - write_start
            print(f"  ✓ Chunk {i+1} written to S3 in {write_time:.2f} seconds ({chunk_rows} rows)")
        except Exception as e:
            print(f"  ✗ Error writing chunk {i+1} to S3: {str(e)}")
        
        # Clean up melted chunk to free memory
        del chunk_melted
        gc.collect()
        
        print(f"  Memory after chunk processing: {get_memory_usage():.2f} GB")
        print(f"  Chunk {i+1} processed in {time.time() - chunk_start_time:.2f} seconds")
    
    total_time = time.time() - start_time
    print(f"All chunks processed and written in {total_time:.2f} seconds")
    print(f"Total rows processed: {total_rows_processed}")
    print(f"Final memory usage: {get_memory_usage():.2f} GB")
    
    return output_base, all_chunk_files, total_rows_processed

# Create timestamp for consistent file naming
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Locate all the enriched volume bids files in S3 using boto3 for better pagination
print("Identifying enriched volume bids files in S3...")
try:
    s3_client = boto3.client('s3')
    bucket = "thesis--ec331-s3"
    prefix = "enriched-volume-bids/"
    enriched_files = []
    
    # Use pagination to handle large number of files
    paginator = s3_client.get_paginator('list_objects_v2')
    page_iterator = paginator.paginate(Bucket=bucket, Prefix=prefix)
    
    for page in page_iterator:
        if 'Contents' in page:
            for obj in page['Contents']:
                file_path = f"s3://{bucket}/{obj['Key']}"
                enriched_files.append(file_path)
    
    print(f"Found {len(enriched_files)} enriched volume bids files")
    
    if not enriched_files:
        raise ValueError("No enriched volume bids files found in S3")
        
except Exception as e:
    print(f"Error listing enriched volume bids files: {str(e)}")
    raise

# Process the files in batches to avoid memory issues
batch_size = 10  # Number of files to process at once
num_batches = (len(enriched_files) + batch_size - 1) // batch_size

print(f"Will process files in {num_batches} batches of up to {batch_size} files each")

# Create a list to store paths of all melted file manifests and all melted files
melted_manifests = []
all_melted_files = []
global_total_rows = 0  # To accumulate total rows processed across batches

# Process each batch
for batch_num in range(num_batches):
    batch_start = batch_num * batch_size
    batch_end = min((batch_num + 1) * batch_size, len(enriched_files))
    batch_files = enriched_files[batch_start:batch_end]
    
    print(f"\nProcessing batch {batch_num + 1}/{num_batches} with {len(batch_files)} files")
    print(f"Memory before batch processing: {get_memory_usage():.2f} GB")
    
    try:
        # Read this batch of files
        print(f"Reading batch of {len(batch_files)} files...")
        batch_start_time = time.time()
        
        # Read the batch of parquet files using dataset=False because batch_files is a list
        batch_df = wr.s3.read_parquet(
            path=batch_files,
            dataset=False
        )
        
        print(f"Read batch in {time.time() - batch_start_time:.2f} seconds")
        print(f"Batch data shape: {batch_df.shape}")
        print(f"Memory after reading batch: {get_memory_usage():.2f} GB")
        
        # Melt and write this batch
        batch_timestamp = f"{timestamp}_batch{batch_num+1:03d}"
        output_base, batch_melted_files, batch_rows = melt_and_write_chunks(batch_df, chunk_size=50000, timestamp=batch_timestamp)
        
        # Add these files to our master list and update the row count
        all_melted_files.extend(batch_melted_files)
        global_total_rows += batch_rows
        
        # Clean up to free memory
        del batch_df
        gc.collect()
        
        # Create a manifest file for this batch
        try:
            manifest = pd.DataFrame({"file_path": batch_melted_files})
            manifest_path = f"{output_base}_manifest.csv"
            wr.s3.to_csv(manifest, manifest_path, index=False)
            print(f"Created manifest file for batch {batch_num + 1}: {manifest_path}")
            
            # Add this manifest to our list
            melted_manifests.append(manifest_path)
        except Exception as e:
            print(f"Error creating manifest file for batch {batch_num + 1}: {str(e)}")
        
    except Exception as e:
        print(f"Error processing batch {batch_num + 1}: {str(e)}")
    
    print(f"Completed batch {batch_num + 1}/{num_batches}")
    print(f"Memory after batch processing: {get_memory_usage():.2f} GB")

print("\nAll batches processed!")
print(f"Total melted files created: {len(all_melted_files)}")

# Create a master manifest of all manifests
try:
    if melted_manifests:
        master_manifest = pd.DataFrame({"manifest_path": melted_manifests})
        master_manifest_path = f"s3://thesis--ec331-s3/melted-volume-bids/master_manifest_{timestamp}.csv"
        wr.s3.to_csv(master_manifest, master_manifest_path, index=False)
        print(f"Created master manifest at: {master_manifest_path}")
except Exception as e:
    print(f"Error creating master manifest: {str(e)}")

# Now, try to create a final DataFrame with all the melted data
print("\nLoading all melted data into final_df...")
try:
    # First, check if it's feasible to load all data at once
    est_rows_per_file = global_total_rows / len(all_melted_files) if all_melted_files else 0
    est_total_rows = est_rows_per_file * len(all_melted_files)
    
    print(f"Estimated total rows in all melted files: {est_total_rows:,.0f}")
    
    # Alternative: Load a sample first to estimate memory requirements
    sample_size = min(5, len(all_melted_files))
    if sample_size > 0:
        print(f"Loading sample of {sample_size} files to estimate memory requirements...")
        sample_df = wr.s3.read_parquet(path=all_melted_files[:sample_size], dataset=False)
        bytes_per_row = sample_df.memory_usage(deep=True).sum() / len(sample_df)
        est_memory_gb = (bytes_per_row * est_total_rows) / 1e9
        
        print(f"Sample loaded: {len(sample_df)} rows")
        print(f"Estimated memory required for full dataset: {est_memory_gb:.2f} GB")
        
        # Show sample data
        print("\nSample data preview:")
        print(sample_df.head())
        
        # Clean up sample
        del sample_df
        gc.collect()
    
    # Load all the data if it seems feasible or force loading
    force_load = True  # Set to True to load everything even if it might be large
    
    if force_load or (est_memory_gb < get_memory_usage() * 5):  # If estimated size is reasonable
        print("\nLoading all melted data...")
        final_df = wr.s3.read_parquet(path=all_melted_files, dataset=False)
        print(f"Successfully loaded all data into final_df")
        print(f"Final dataframe shape: {final_df.shape}")
        print(f"Final dataframe columns: {final_df.columns.tolist()}")
        print(f"Final dataframe preview:\n{final_df.head()}")
        
        # Optional: Save the final DataFrame to a single file for easier access later
        final_output_path = f"s3://thesis--ec331-s3/melted-volume-bids/combined_melted_data_{timestamp}.parquet"
        print(f"\nSaving final_df to {final_output_path}...")
        wr.s3.to_parquet(
            df=final_df,
            path=final_output_path,
            index=False,
            compression="snappy"
        )
        print(f"Successfully saved final_df to {final_output_path}")
    else:
        print(f"\nWarning: Full dataset would require approximately {est_memory_gb:.2f} GB of memory.")
        print("To load all data, set force_load = True in the code.")
        print(f"You can still access all the individual melted files from {master_manifest_path}")
        
except Exception as e:
    print(f"Error loading melted data: {str(e)}")
    print("The melted data is still available in the individual files.")
    print(f"You can find all file paths in the master manifest: {master_manifest_path}")

print("\nProcess complete!")
print(f"Final memory usage: {get_memory_usage():.2f} GB")

Initial memory usage: 0.29 GB
Identifying enriched volume bids files in S3...
Found 2123 enriched volume bids files
Will process files in 213 batches of up to 10 files each

Processing batch 1/213 with 10 files
Memory before batch processing: 0.29 GB
Reading batch of 10 files...
Read batch in 0.74 seconds
Batch data shape: (100000, 47)
Memory after reading batch: 0.38 GB
Starting streaming melt of dataframe with shape: (100000, 47)
Current memory usage: 0.38 GB
Processing in 2 chunks of size 50000
Processing chunk 1/2 (rows 0 to 49999)
Memory before chunk processing: 0.38 GB
  ✓ Chunk 1 written to S3 in 1.37 seconds (500000 rows)
  Memory after chunk processing: 0.43 GB
  Chunk 1 processed in 3.66 seconds
Processing chunk 2/2 (rows 50000 to 99999)
Memory before chunk processing: 0.43 GB
  ✓ Chunk 2 written to S3 in 1.34 seconds (500000 rows)
  Memory after chunk processing: 0.44 GB
  Chunk 2 processed in 3.47 seconds
All chunks processed and written in 7.29 seconds
Total rows processed

In [4]:
import awswrangler as wr
files = wr.s3.list_objects("s3://thesis--ec331-s3/enriched-volume-bids/")
print(f"Found {len(files)} files")

Found 2123 files


In [ ]:
import pandas as pd
import awswrangler as wr
import time
from datetime import datetime
import gc
import os
import psutil

# List of band columns we want to melt
band_cols = [f"BANDAVAIL{i}" for i in range(1, 11)]

def get_memory_usage():
    """Return the current memory usage of the process in GB"""
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024 / 1024 / 1024
    return memory_gb

def melt_and_write_chunks(df, chunk_size=50000):
    """
    Melts the BANDAVAIL columns in chunks and writes each chunk directly to S3
    """
    print(f"Starting streaming melt of dataframe with shape: {df.shape}")
    print(f"Initial memory usage: {get_memory_usage():.2f} GB")
    start_time = time.time()
    
    # Calculate number of chunks
    num_chunks = (len(df) + chunk_size - 1) // chunk_size
    print(f"Processing in {num_chunks} chunks of size {chunk_size}")
    
    # Create a timestamp for the file prefix
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    output_base = f"s3://thesis--ec331-s3/melted-volume-bids/enriched_volume_bids_melted_{timestamp}"
    
    # Track total rows processed
    total_rows_processed = 0
    
    # Process dataframe in chunks
    for i in range(num_chunks):
        chunk_start = i * chunk_size
        chunk_end = min((i + 1) * chunk_size, len(df))
        
        print(f"Processing chunk {i+1}/{num_chunks} (rows {chunk_start} to {chunk_end-1})")
        print(f"Memory before chunk processing: {get_memory_usage():.2f} GB")
        
        # Extract chunk and immediately release the original slice reference
        chunk = df.iloc[chunk_start:chunk_end].copy()
        
        # Identify columns not being melted
        id_vars_cols = [col for col in chunk.columns if col not in band_cols]
        
        # Keep only necessary columns
        chunk = chunk[id_vars_cols + [col for col in band_cols if col in chunk.columns]]
        
        # Melt this chunk
        chunk_start_time = time.time()
        chunk_melted = pd.melt(
            chunk,
            id_vars=id_vars_cols,
            value_vars=[col for col in band_cols if col in chunk.columns],
            var_name="BIDBAND",
            value_name="BIDVOLUME"
        )
        
        # Clean up original chunk to free memory
        del chunk
        gc.collect()
        
        # Extract the band number
        chunk_melted["BIDBAND"] = chunk_melted["BIDBAND"].str.extract(r"BANDAVAIL(\d+)").astype(int)
        
        # Filter out null values to reduce size
        chunk_melted = chunk_melted.dropna(subset=["BIDVOLUME"])
        
        # Count rows in this chunk
        chunk_rows = len(chunk_melted)
        total_rows_processed += chunk_rows
        
        # Write this chunk directly to S3
        chunk_output = f"{output_base}_part{i+1:04d}.parquet"
        write_start = time.time()
        
        try:
            wr.s3.to_parquet(
                df=chunk_melted,
                path=chunk_output,
                index=False,
                compression="snappy"
            )
            write_time = time.time() - write_start
            print(f"  ✓ Chunk {i+1} written to S3 in {write_time:.2f} seconds ({chunk_rows} rows)")
        except Exception as e:
            print(f"  ✗ Error writing chunk {i+1} to S3: {str(e)}")
        
        # Clean up melted chunk to free memory
        del chunk_melted
        gc.collect()
        
        print(f"  Memory after chunk processing: {get_memory_usage():.2f} GB")
        print(f"  Chunk {i+1} processed in {time.time() - chunk_start_time:.2f} seconds")
    
    total_time = time.time() - start_time
    print(f"All chunks processed and written in {total_time:.2f} seconds")
    print(f"Total rows processed: {total_rows_processed}")
    print(f"Final memory usage: {get_memory_usage():.2f} GB")
    
    return output_base

# Now let's apply this to the enriched_df
if 'enriched_df' in locals():
    print("Streaming melt of the enriched dataframe...")
    
    # Try to install psutil if not already installed
    try:
        import psutil
    except ImportError:
        print("Installing psutil package for memory monitoring...")
        import sys
        !{sys.executable} -m pip install psutil
        import psutil
    
    # Melt and write the dataframe using the streaming approach
    output_base = melt_and_write_chunks(enriched_df, chunk_size=50000)
    
    print(f"\nSummary:")
    print(f"Successfully melted and wrote dataframe to S3")
    print(f"Base path: {output_base}")
    print(f"Check S3 console or use AWS CLI to view all files")
    
    # Optional: Create a manifest file with the locations of all chunks
    try:
        # List all the chunks we created
        pattern = f"{output_base}_part*.parquet"
        chunk_files = wr.s3.list_objects(pattern)
        
        # Write a simple manifest file
        manifest = pd.DataFrame({"file_path": chunk_files})
        manifest_path = f"{output_base}_manifest.csv"
        wr.s3.to_csv(manifest, manifest_path, index=False)
        print(f"Created manifest file with paths to all {len(chunk_files)} chunks: {manifest_path}")
    except Exception as e:
        print(f"Error creating manifest file: {str(e)}")
    
else:
    print("The enriched_df is not available. Please run the enrichment code first.")

In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta

# Define which columns to melt
price_band_cols = [f"PRICEBAND{i}" for i in range(1, 11)]

def melt_one_price_file(input_path, output_dir):
    """
    Reads one Parquet file (price bids) into memory,
    melts the PRICEBAND columns into a single column,
    and saves the melted data to a new Parquet file in output_dir.
    """
    try:
        # 1) Read the single Parquet file into a pandas DataFrame
        df = pd.read_parquet(input_path)
        
        # 2) Identify all other columns as id_vars
        id_vars_cols = [col for col in df.columns if col not in price_band_cols]
        
        # 3) Melt the DataFrame
        df_long = pd.melt(
            df,
            id_vars=id_vars_cols,
            value_vars=price_band_cols,
            var_name="BIDBAND",
            value_name="BIDPRICE"
        )
        
        # 4) Convert "BIDBAND" from e.g. "PRICEBAND1" -> integer 1
        df_long["BIDBAND"] = df_long["BIDBAND"].str.extract(r"(\d+)").astype(int)
        
        # 5) (Optional) Convert other columns to consistent dtypes
        #    Adjust column names if they differ in your data
        if "BIDTYPE" in df_long.columns:
            df_long["BIDTYPE"] = df_long["BIDTYPE"].astype(str)
            
        if "DUID" in df_long.columns:
            df_long["DUID"] = df_long["DUID"].astype(str)
            
        if "SETTLEMENTDATE" in df_long.columns:
            df_long["SETTLEMENTDATE"] = pd.to_datetime(df_long["SETTLEMENTDATE"])
            # Add 4 hours, 5 minutes offset for APPLICABLEFROM
            df_long["APPLICABLEFROM"] = df_long["SETTLEMENTDATE"] + timedelta(hours=4, minutes=5)
            df_long["APPLICABLEFROM"] = pd.to_datetime(df_long["APPLICABLEFROM"])
        
        # 6) Construct the output file path
        base_name = os.path.basename(input_path)
        melted_name = base_name.replace(".parquet", "_melted.parquet")
        output_path = os.path.join(output_dir, melted_name)
        
        # 7) Save melted data to a new Parquet file
        df_long.to_parquet(output_path, index=False)
        print(f"Melted file saved to {output_path}")
        return True
    except Exception as e:
        print(f"Error processing {input_path}: {str(e)}")
        return False

def main():
    input_dir = "/Volumes/T7/bid-price-data-excluding-energy-B2"  # Where your price Parquet files live
    output_dir = "/Volumes/T7/bid-price-melted-files"  # Where to save melted files
    os.makedirs(output_dir, exist_ok=True)
    
    # Gather Parquet files
    parquet_files = sorted(glob.glob(os.path.join(input_dir, "*.parquet")))
    
    # Exclude hidden macOS files
    parquet_files = [f for f in parquet_files if not os.path.basename(f).startswith("._")]
    print(f"Found {len(parquet_files)} Parquet files in {input_dir}.")
    
    # Process each file individually
    successful = 0
    for i, file_path in enumerate(parquet_files):
        print(f"Processing file {i+1}/{len(parquet_files)}: {os.path.basename(file_path)}")
        if melt_one_price_file(file_path, output_dir):
            successful += 1
        
        # Print progress every 10 files
        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(parquet_files)} files, {successful} successful")
    
    print(f"All price files processed. Successfully melted {successful} out of {len(parquet_files)} files.")
    
    # Verify output files
    output_files = glob.glob(os.path.join(output_dir, "*_melted.parquet"))
    print(f"\nVerified {len(output_files)} melted files in the output directory.")
    
    if len(output_files) > 0:
        # Optionally, load and print a sample from the first output file
        try:
            sample_file = output_files[0]
            sample_df = pd.read_parquet(sample_file)
            print(f"\nSample from first melted file ({os.path.basename(sample_file)}):")
            print(sample_df.head())
            print(f"Sample file shape: {sample_df.shape}")
        except Exception as e:
            print(f"Error reading sample file: {str(e)}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import awswrangler as wr
import time

# Add timing and logging
print("Starting data load...")
start_time = time.time()

# Read all Parquet files from the S3 folder - UNCOMMENT THIS
s3_path = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Volume-Bids/PUBLIC_DVD_BIDPEROFFER1_202311010000.parquet/"
df = wr.s3.read_parquet(path=s3_path)

print(f"Data loaded in {time.time() - start_time:.2f} seconds")
print(f"DataFrame shape: {df.shape}")
print(f"Memory usage: {df.memory_usage().sum() / 1024 / 1024:.2f} MB")

# Print some sample data to understand structure
print("\nSample data:")
sample = df.sample(3)
print(sample[["DUID", "TRADINGDATE", "PERIODID", "MAXAVAIL"] + 
      [f"PRICEBAND{i}" for i in range(1, 11)] + 
      [f"BANDAVAIL{i}" for i in range(1, 11)]])

# Define a truly vectorized approach for capping
def cap_bands_vectorized(df_group):
    """Process a group of bids with the same DUID, TRADINGDATE, PERIODID"""
    # Create a copy to avoid modifying the original
    result_df = df_group.copy()
    
    # For each row in the group
    for idx, row in result_df.iterrows():
        max_avail = row["MAXAVAIL"]
        
        # Extract prices and volumes
        prices = [row[f"PRICEBAND{i}"] if f"PRICEBAND{i}" in row and not pd.isna(row[f"PRICEBAND{i}"]) else np.nan 
                 for i in range(1, 11)]
        volumes = [row[f"BANDAVAIL{i}"] if f"BANDAVAIL{i}" in row and not pd.isna(row[f"BANDAVAIL{i}"]) else 0.0 
                  for i in range(1, 11)]
        
        # Create a list of (price, volume, band_index)
        bands = [(p, v, i) for i, (p, v) in enumerate(zip(prices, volumes), 1) if not pd.isna(p)]
        
        # Sort by price
        bands.sort(key=lambda x: x[0])
        
        # Apply capping
        running_sum = 0.0
        capped_volumes = [0.0] * 10
        
        for price, volume, band_idx in bands:
            if running_sum >= max_avail:
                break
                
            remaining = max_avail - running_sum
            capped_volume = min(volume, remaining)
            
            capped_volumes[band_idx-1] = capped_volume
            running_sum += capped_volume
        
        # Update the result dataframe
        for i in range(1, 11):
            col = f"BANDAVAIL{i}"
            if col in result_df.columns:
                result_df.at[idx, col] = capped_volumes[i-1]
    
    return result_df

# Much more efficient approach
def process_data(df, chunk_size=1000):
    print("\nProcessing data in chunks...")
    capped_dfs = []
    total_chunks = (len(df) + chunk_size - 1) // chunk_size
    
    for i in range(0, len(df), chunk_size):
        chunk_start = time.time()
        print(f"Processing chunk {i//chunk_size + 1}/{total_chunks}")
        
        # Get chunk
        chunk = df.iloc[i:i+chunk_size].copy()
        
        # Process chunk - group by key columns
        grouped = chunk.groupby(["DUID", "TRADINGDATE", "PERIODID"])
        capped_chunk = grouped.apply(cap_bands_vectorized)
        
        capped_dfs.append(capped_chunk)
        print(f"Chunk {i//chunk_size + 1} completed in {time.time() - chunk_start:.2f} seconds")
    
    # Combine results
    if capped_dfs:
        return pd.concat(capped_dfs)
    else:
        return pd.DataFrame()

# Execute the processing
capped_df = process_data(df, chunk_size=500)
print(f"\nAll processing completed in {time.time() - start_time:.2f} seconds")
print(capped_df.head())